# Des syntagmes aux phrases

Deux grammaires sont utilisées ici :

* `./data` — le syntagme nominal seul (`D NOM`, `D ADJ NOM`, `D NOM ADJ`)
* `./data3` — la phrase complète, avec des verbes de quatre valences

## Le point de départ : `D NOM`

Il reste de l'ambiguïté : le pluriel français s'aligne sur deux nombres kalaba
(duel et pluriel), d'où les deux traductions.

In [1]:
from pfmg.parsing.main.actions import parsing_action

parsing_action(
    {
        "data": "des garçons",
        "path": "./data",
        "keep": "all",
    }
)

tulo zo
tulol zoj


## Trois constituants : `det adj nom`

Pour enchaîner plusieurs syntagmes, on construit le parseur une seule fois
(`KParser.from_yaml`) au lieu de repasser par `parsing_action`.

In [2]:
from pfmg.parsing.parser import KParser

parser = KParser.from_yaml("./data")
parser.parse(data="des petites autruches", keep="all")

['siprug kizin zij', 'sipru kizi zi']

L'adjectif peut aussi être postposé : `les autruches vertes` passe par la règle
`D NOM ADJ` et aboutit au même ordre kalaba `NOM ADJ D`.

In [3]:
phrases = [
    "la petite autruche",
    "des petites autruches",
    "les autruches vertes",
    "le gentil garçon",
    "les gentilles filles",
]

{phrase: parser.parse(data=phrase, keep="all") for phrase in phrases}

{'la petite autruche': ['sipruj kizir ris'],
 'des petites autruches': ['siprug kizin zij', 'sipru kizi zi'],
 'les autruches vertes': ['siprug regin rij', 'sipru regi ri'],
 'le gentil garçon': ['tulov zontar ros'],
 'les gentilles filles': ['grig zontin rij', 'gri zonti ri']}

## L'accord filtre la combinatoire

Sans accord, un syntagme à trois constituants multiplierait les analyses (trois
genres kalaba possibles pour le déterminant × trois pour l'adjectif). L'accord
`Genre,Nombre` élimine ces analyses, et rejette aussi les syntagmes mal accordés
en français : aucune analyse ne survit pour `des petit autruches`.

In [4]:
parser.parse(data="des petit autruches", keep="all")

[]

## La phrase complète (`./data3`)

`data3` ajoute trois choses au syntagme nominal :

* une catégorie `V` dont le trait lexical `Val` porte la **valence** —
  `intr`, `tdir`, `tind`, `ditr` ;
* une catégorie `P` pour les prépositions, et une règle `PP` qui les transforme
  en postpositions kalaba ;
* une catégorie `NUM` : `deux` est lexicalement **duel** en kalaba, `trois` et
  `quatre` sont pluriels.

Le français est SVO, le kalaba est SOV : le verbe part en fin de proposition.

Chaque règle de phrase accorde ses constituants séparément, avec un segment par
constituant : `"Nombre;Nombre,Val=tdir;"` fait accorder le sujet et le verbe en
nombre, impose un verbe transitif direct, et laisse l'objet libre.

In [5]:
parser3 = KParser.from_yaml("./data3")

parser3.parse(data="les petits enfants mangent deux autruches jaunes", keep="all")

['telaazi kizerak rek sipruazi gelirak duz nagetak',
 'telag kizen rej sipruazi gelirak duz nagetan']

Les deux analyses ne diffèrent que par le nombre du sujet : `les` est pluriel en
français là où le kalaba distingue un duel (`telaazi`) d'un pluriel (`telag`).
L'objet, lui, est duel dans les deux cas, imposé par `deux`.

## Les quatre valences

C'est l'accord de la règle qui sélectionne le verbe, pas l'inverse.

In [6]:
valences = [
    "le petit enfant dort",  # intransitif
    "la fille voit deux autruches",  # transitif direct
    "les enfants parlent à la maman",  # transitif indirect
    "la maman donne une banane à la fille",  # ditransitif
]

{phrase: parser3.parse(data=phrase, keep="all") for phrase in valences}

{'le petit enfant dort': ['telaj kizer res zomta'],
 'la fille voit deux autruches': ['grij ris sipruazi duz sepita'],
 'les enfants parlent à la maman': ['telag rej mimav ris ka lubitan',
  'telaazi rek mimav ris ka lubitak'],
 'la maman donne une banane à la fille': ['mimav ris ninov zes grij ris ka mizuta']}

## Ce que la grammaire refuse

Une valence mal servie ou un sujet mal accordé ne laisse aucune analyse.

In [7]:
refus = [
    "les enfants mangent",  # transitif direct sans objet
    "le garçon dort la banane",  # intransitif avec objet
    "les enfants parlent la maman",  # transitif indirect sans préposition
    "le enfant mangent deux autruches",  # sujet et verbe désaccordés
]

{phrase: parser3.parse(data=phrase, keep="all") for phrase in refus}

{'les enfants mangent': [],
 'le garçon dort la banane': [],
 'les enfants parlent la maman': [],
 'le enfant mangent deux autruches': []}